**This script creates the candidate site to be used for clustering from a generic turbine dataset**

In [ ]:
#Importing libraries
import numpy as np
import pandas as pd

**preprocessing the data**

In [ ]:
# creating a candidate site
excluded_nodes = {'stationary_hub', 'rotating_hub', 'blade_maxchord', 'tower_top'}


GT_name = 'GT10'
my_path = 'add_path_here_for_GT'

GT_df = pd.read_csv(f'{my_path}{GT_name}_DB_list.csv')
GT_df = GT_df[~GT_df['node'].isin(excluded_nodes)]

# use only one turbulence seed
seed = GT_df['seedID'].unique()[0]
GT_df = GT_df[GT_df['seedID'] == seed]

env_inputs = [
    'windSpeed',
    'iRef',
    'shearExp',
    'density',
    'inFlowAngle'
]
site_data_path = 'add_path_to_candidate_site_here'
node_features = ['blade_root_Mx_4', 'blade_root_Mx_9', 'blade_root_Mx_10', 'blade_root_Mx_14','blade_root_My_4', 'blade_root_My_9', 'blade_root_My_10', 'blade_root_My_14', 'tower_base_My_4', 'tower_base_My_9']


In [ ]:
def normalise(GT_df):
    #------------------------------------------------------damage normalisation function---------------------------------------------------------------------
    #baseline values
    windspeed_base = 16
    iref_base = 0.14
    density_base = 1.15
    inflow_base = 0
    shear_base = 0.15

    #averaging across turbulence seeds for the damage does not work as they need to be raised to the power of the wohler exp first.
    def power_average_damage(damage_values, wohler):
        return (np.mean(damage_values ** wohler)) ** (1/wohler)


    #finding the damage value that matches the baseline environmental conditions
    #this creates a df with a corresponding baseline damage for each subset
    baseline_rows = GT_df.loc[
        (GT_df['windSpeed'] == windspeed_base) &
        (GT_df['iRef'] == iref_base) &
        (GT_df['density'] == density_base) &
        (GT_df['shearExp'] == shear_base) &
        (GT_df['inFlowAngle'] == inflow_base),
        ['node', 'load', 'wohler', 'damage']
    ]

    # apply power average across the three seeds for each subset
    #the lambda x creates another function to handle each 3 identical rows seperately, where x is each group
    baseline = baseline_rows.groupby(['node', 'load', 'wohler']).apply(
        lambda x: pd.Series({
            'baseline_damage': power_average_damage(x['damage'].values, x.name[2])
        }) ,
        include_groups=False
    ).reset_index()

    print(baseline)
    print(baseline.groupby(['node', 'load', 'wohler']).size())  # should be 1 per combination

    # merge - now one baseline value per subset, no duplicates
    GT_df = GT_df.merge(baseline, on=['node', 'load', 'wohler'], how='left')

    GT_df['damage'] = GT_df['damage'] / GT_df['baseline_damage']

    print(GT_df['damage'].describe())

    return GT_df


In [ ]:
GT_df = normalise(GT_df)

seed = GT_df['seedID'].unique()[0]
GT_df = GT_df[GT_df['seedID'] == seed]

# get one row per unique environmental condition with damage for each subset
# pivot so each unique env condition is one row with damage columns per subset
GT_wide = GT_df.pivot_table(
    index=['windSpeed', 'iRef', 'shearExp', 'density', 'inFlowAngle'],
    columns=['node', 'load', 'wohler'],
    values='damage'
).reset_index()

# flatten column names
GT_wide.columns = ['_'.join([str(c) for c in col]).strip('_') if col[1] != '' 
                   else col[0] for col in GT_wide.columns]


**creating a candidate site**

This site has high turbulence on the left and high shear on the right, this gradient variation is applied and then used to select from the GT dataset.

In [ ]:
n_rows = 5
n_cols = 10

windSpeed_vals = [8, 10, 12, 14, 16, 18] #only a 50% reduction
iRef_vals = [0.08, 0.1,  0.12, 0.14, 0.16, 0.18, 0.2 ]
shear_vals = [0.05, 0.15, 0.25]
inflow_vals = [-8, 0, 8]
density_vals = [1.25]

def nearest(val, options):
    return min(options, key=lambda x: abs(x - val))

np.random.seed(42)
turbines = []

seen_conditions = set()

for row in range(n_rows):
    for col in range(n_cols):
        x = col / (n_cols - 1)
        y = row / (n_rows - 1)

        iRef_step = iRef_vals[1] - iRef_vals[0]
        shear_step = shear_vals[1] - shear_vals[0]
        wind_step = windSpeed_vals[1] -  windSpeed_vals[0]
        inflow_step = inflow_vals[1] -  inflow_vals[0]

        k_weibull = 2.0  # shape parameter
        lambda_weibull = 10.0  # scale parameter (related to mean wind speed)


        attempts = 0
        while True:
            iRef = nearest(0.08 + x * 0.12 + np.random.uniform(-iRef_step, iRef_step), iRef_vals)
            windSpeed = nearest(4 + (y) * 24 + np.random.uniform(-wind_step, wind_step), windSpeed_vals)
            shear = nearest(0.05 + (1-x) * 0.2 + np.random.uniform(-shear_step, shear_step), shear_vals)  # candidate site 3
            inFlowAngle = nearest(-8 + (1-y) * 8 + np.random.uniform(-inflow_step, inflow_step), inflow_vals)  # candidate site 3
            #windSpeed = nearest(np.random.weibull(k_weibull) * lambda_weibull, windSpeed_vals)  #candidate site 2
            #inFlowAngle = np.random.choice(inflow_vals)
            density = np.random.choice(density_vals)

            condition = (round(windSpeed, 3), round(iRef, 3), round(shear, 3), round(density, 3), round(inFlowAngle, 3))

            # check exists in GT_wide and is not a duplicate
            match = GT_wide[
                (GT_wide['windSpeed'].round(3) == condition[0]) &
                (GT_wide['iRef'].round(3) == condition[1]) &
                (GT_wide['shearExp'].round(3) == condition[2]) &
                (GT_wide['density'].round(3) == condition[3]) &
                (GT_wide['inFlowAngle'].round(3) == condition[4])
            ]

            if len(match) > 0 and condition not in seen_conditions:
                seen_conditions.add(condition)
                break

            attempts += 1
            if attempts > 1000:
                print(f'Warning: could not find valid unique condition for row {row} col {col}')
                break
        
        print(windSpeed)
        
        turbines.append({
            'turbine_id': f'T{str(len(turbines)+1).zfill(3)}',
            'grid_row': row,
            'grid_col': col,
            'windSpeed': windSpeed,
            'iRef': iRef,
            'shearExp': shear,
            'inFlowAngle': inFlowAngle,
            'density': density
        })

candidate_site = pd.DataFrame(turbines)

for col in env_inputs:
    candidate_site[col] = candidate_site[col].round(4)
    GT_wide[col] = GT_wide[col].round(4)

# merge with GT_wide to get actual damage
candidate_site = candidate_site.merge(
    GT_wide,
    on=env_inputs,
    how='left'
)

print(f'Candidate site: {len(candidate_site)} turbines')
print(f'NaN damage values: {candidate_site[node_features].isna().sum().sum()}')
candidate_site.to_csv(f'{site_data_path}candidate_site5.csv', index=False)


In [ ]:
# check which turbines have NaN damage
nan_turbines = candidate_site[candidate_site[node_features].isna().any(axis=1)]
print(f'Turbines with NaN: {len(nan_turbines)}')
print(nan_turbines[env_inputs])

# check if those env conditions exist in GT_wide
for _, row in nan_turbines.iterrows():
    match = GT_wide[
        (GT_wide['windSpeed'] == row['windSpeed']) &
        (GT_wide['iRef'] == row['iRef']) &
        (GT_wide['shearExp'] == row['shearExp']) &
        (GT_wide['density'] == row['density']) &
        (GT_wide['inFlowAngle'] == row['inFlowAngle'])
    ]
    print(f"T{row['turbine_id']} - {row[env_inputs].tolist()} - matches: {len(match)}")

#ensuring there are no duplicates
duplicates = candidate_site.duplicated(subset=env_inputs)
print(f'Number of duplicate environmental conditions: {duplicates.sum()}')
print(candidate_site[duplicates][env_inputs])   